In this notebook i will be deploying a yt chatbot using gradio. The chatbot takes the video url as input

Now you are good to go with asking any questions to the chatbot about the video and it will answer. The chatbot can help clear doubts about specific topics discussed in the video, can give example/analogies to help better understand the topics discussed

In [24]:
import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

# imports and installations


In [56]:
!pip install -q youtube-transcript-api langchain-community faiss-cpu tiktoken python-dotenv

In [137]:
!pip install -q groq
!pip install -q sentence-transformers faiss-cpu
!pip install -q youtube-transcript-api
!pip install -q gradio

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [2]:
from google.colab import userdata
GROQ_API=userdata.get('Groq_API')

In [3]:
from groq import Groq
client = Groq(api_key=GROQ_API)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Video ID extractor

In [4]:
from urllib.parse import urlparse, parse_qs
import re

def extract_video_id_from_url(url):
    try:
        parsed_url = urlparse(url)

        # Case 1: youtu.be/<id>
        if parsed_url.hostname == 'youtu.be':
            return parsed_url.path[1:]

        # Case 2: youtube.com/watch?v=<id>
        if parsed_url.hostname in ['www.youtube.com', 'youtube.com', 'm.youtube.com']:
            if parsed_url.path == '/watch':
                return parse_qs(parsed_url.query).get('v', [None])[0]

            # Case 3: /embed/<id>
            if parsed_url.path.startswith('/embed/'):
                return parsed_url.path.split('/')[2]

            # Case 4: /shorts/<id>
            if parsed_url.path.startswith('/shorts/'):
                return parsed_url.path.split('/')[2]

        return None

    except:
        return None

# RAG part

## Get transcript

In [25]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

def get_transcript(video_id):

  try:
      # Create an instance of YouTubeTranscriptApi to call fetch on it, addressing the 'missing self' error.
      yt_api_instance = YouTubeTranscriptApi()
      transcript_list = yt_api_instance.fetch(video_id=video_id, languages=["en"])

      # Flatten it to plain text, accessing the 'text' attribute of each object.
      transcript = " ".join(chunk.text for chunk in transcript_list)

  except TranscriptsDisabled:
      print("No captions available for this video.")
  except Exception as e:
      print(f"An unexpected error occurred: {e}")
  return transcript

## chunking

In [40]:
!pip install -q langchain

In [41]:
!pip install -q langchain-text-splitters

In [42]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

Dynamic chunking based on the transcript lenth, large chunks for larhge video

and recursive because i feel it is the best type of splitters as of now. (In future i may change this to semantic text splitter)

In [57]:
def dynamic_chunking(transcript):

    length = len(transcript)

    # Dynamic sizing
    if length < 5000:
        chunk_size = 500
        chunk_overlap = 50
    elif length < 20000:
        chunk_size = 800
        chunk_overlap = 100
    else:
        chunk_size = 1200
        chunk_overlap = 150

    splitter= RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    chunks= splitter.create_documents([transcript])

    return chunks

## FAISS vector store

In [54]:
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [71]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def create_vector_store(chunks):
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        encode_kwargs={"normalize_embeddings": True},

    )
    vector_store= FAISS.from_documents(chunks,embeddings)
    return vector_store


## retrieval

In [93]:
def retrieve(query, vector_store, top_k=None):
    total_chunks=len(vector_store.docstore._dict)
    if top_k is None:
        if total_chunks < 10:
            k = 3
        elif total_chunks < 30:
            k = 5
        elif total_chunks < 80:
            k = 8
        else:
            k = 10
    else:
        k = top_k

    retriever=vector_store.as_retriever(search_type= "similarity",search_kwargs={"k":k})
    docs = retriever.invoke(query)

    return "\n".join([doc.page_content for doc in docs])

## asking llm

In [ ]:
from groq import Groq
# Groq LLM
client = Groq(api_key=GROQ_API)
MODEL_NAME = "mixtral-8x7b-32768"  # Mistral 7x8B 32K

In [77]:
from langchain_core.prompts import PromptTemplate


In [153]:
def ask_llm(query, context, chat_history):

    system_prompt = """
You are a helpful AI assistant answering questions about a YouTube video.

Use the provided transcript context as your primary source of truth.

If the question is directly about the video content:
- Answer using the transcript.
- You may elaborate, simplify, or provide analogies using your own knowledge,
  as long as they remain consistent with the video's topic.

If the transcript does not contain the answer:
- You may use your general knowledge to answer.
- Clearly indicate when you are expanding beyond the transcript.

Do not fabricate specific claims about the video that are not supported by the transcript.
Be accurate, helpful, and clear.
"""

    # Build conversation messages
    messages = [{"role": "system", "content": system_prompt}]

    # Add previous history
    for user_msg, bot_msg in chat_history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    # Add current query WITH context
    user_prompt = f"""
Context:
{context}

Question:
{query}
"""
    messages.append({"role": "user", "content": user_prompt})

    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
    )

    return completion.choices[0].message.content

In [155]:
def rag_pipeline(url, query, chat_history):

    video_id = extract_video_id_from_url(url)
    if not video_id:
        return "Video not found."

    transcript = get_transcript(video_id)
    chunks = dynamic_chunking(transcript)
    vector_store = create_vector_store(chunks)

    context = retrieve(query, vector_store)

    answer = ask_llm(query, context, chat_history)

    return answer

# Gradio UI

In [158]:
import gradio as gr

def chat_interface(url, query, chat_history):

    if chat_history is None:
        chat_history = []

    answer = rag_pipeline(url, query, chat_history)

    chat_history.append((query, answer))

    return chat_history, chat_history


with gr.Blocks() as demo:

    gr.Markdown("# 🎥 YouTube RAG Chatbot with Memory")

    url_input = gr.Textbox(label="YouTube URL")

    chatbot = gr.Chatbot()
    state = gr.State([])

    query_input = gr.Textbox(label="Ask your question")

    submit_btn = gr.Button("Ask")

    submit_btn.click(
        chat_interface,
        inputs=[url_input, query_input, state],
        outputs=[chatbot, state],
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7322c5e03476ac3e54.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
